# GeoAtt-PointNet++ v5.0.0 — Low-Data Regime Training + Evaluation

**Hipotesis**: GeoAtt menyediakan inductive bias yang menguntungkan dalam regime enrollment terbatas (1 sampel per sesi).

**Setup**:
- **2 varian**: `no_geom` (PointNet++ murni) vs `with_geom` (GeoAtt full)
- **10 subjek** (gede di-drop, <15 sesi valid)
- **Split deterministic chronological**: 8 train / 2 val / 2 test / 3 holdout per subjek (150 frames total)
- **1 median frame per sesi** (low-data regime)
- **Triplet batch-hard loss** dengan margin 0.3
- **Auxiliary classification loss** (force geom branch belajar)
- **Val pair EER** untuk model selection (bukan val_loss)
- **Fixed budget**: 120 + 30 epoch, no early stopping
- **Augmentation depth-focused**: rotation ±45°, tilt ±20°, XY+Z translation ±3cm, scale 0.95-1.05
- **10 seeds**: 42, 123, 2024, 7, 31337, 0, 1, 2, 3, 4

**Auto-tune VRAM tier** (target ~90% GPU utilization, max RAM cache):

| Tier | VRAM | BS | N_points | AMP | Preload | Repeat | Aug variants/epoch |
|------|------|----|----|----|----|----|----|
| **T3** | ≥90 GB (H100/A100 96GB) | 80 | 16384 | bf16 | ✅ | 20 | 1600 |
| **T2** | ≥80 GB (A100 80GB) | 80 | 16384 | bf16 | ✅ | 16 | 1280 |
| **T1** | ≥40 GB (A100 40GB / L4) | 64 | 12288 | bf16 | ✅ | 12 | 960 |
| **T0** | <40 GB (T4 / V100) | 32 | 8192 | fp16 | ❌ | 5 | 400 |

**Strategi**: dataset kecil (80 train frames) → maksimalkan compute per-iteration via N_POINTS lebih besar + REPEAT lebih tinggi + preload ke RAM. Hasil: GPU util naik dari ~30-40% → ~85-95%.

**Gate 0**: sanity baseline geom-only LogReg = 98.7% (PASSED, lanjut full run).

## 1. Setup — GitHub Clone + Dataset in Repo

**Prerequisite**: simpan GitHub Personal Access Token sebagai Colab Secret:
1. Sidebar kiri → 🔑 Secrets → Add Secret
2. Name: `GITHUB_TOKEN`, Value: token dari https://github.com/settings/tokens
3. Toggle "Notebook access" ON

Dataset (`3DCNN/dataset/`) sudah include di repo — tidak perlu Google Drive mount lagi.

In [ ]:
from google.colab import userdata
import os, sys, subprocess
from pathlib import Path

# ---- 1. Dataset dari repo (sudah include di GitHub) ----
# ---- 2. Clone repo dari GitHub ----
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_URL = f'https://{GITHUB_TOKEN}@github.com/RZulfikri/Thesis.git'
REPO_DIR = Path('/content/Thesis')
PROJECT_ROOT = REPO_DIR / '3DCNN'
COLAB_BRANCH = 'colab'

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin

# ---- 3. Checkout branch 'colab' ----
os.chdir(str(REPO_DIR))
# Coba checkout existing colab branch, atau buat dari main
ret = subprocess.run(['git', 'checkout', COLAB_BRANCH], capture_output=True)
if ret.returncode != 0:
    # Branch belum ada — buat dari main
    !git checkout -b {COLAB_BRANCH}
    print(f'Created new branch: {COLAB_BRANCH}')
else:
    # Branch sudah ada — merge latest main
    !git merge origin/main --no-edit --strategy-option theirs 2>/dev/null || true
    print(f'Checked out existing branch: {COLAB_BRANCH}')

# Git identity untuk commit
!git config user.email "colab-runner@thesis.local"
!git config user.name "Colab Runner"

# ---- 4. Dataset dari repo (sudah include di GitHub) ----
DATASET_DST = PROJECT_ROOT / 'dataset'

if DATASET_DST.exists():
    print(f'Dataset ditemukan di repo: {DATASET_DST}')
else:
    print(f'\u26a0\ufe0f Dataset tidak ditemukan di repo: {DATASET_DST}')
    print('Pastikan repo di-clone dengan dataset (branch colab).')

# ---- 5. Setup Python path ----
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(str(PROJECT_ROOT))

!ls -la | head -20
print(f'\nPROJECT_ROOT : {PROJECT_ROOT}')
print(f'Branch       : {COLAB_BRANCH}')
print(f'Dataset      : {DATASET_DST}')


In [ ]:
# Verifikasi modules
!python -c "from utils.geometry_schema import GEOMETRY_DIM; assert GEOMETRY_DIM == 13; print('GEOMETRY_DIM =', GEOMETRY_DIM)"
!python -c "from utils.dataset_lowdata import DROPPED_SUBJECTS; print('DROPPED:', DROPPED_SUBJECTS)"

# Clear pycache
!find . -name __pycache__ -type d -exec rm -rf {} + 2>/dev/null
print('\u2705 Setup OK')


## 2. Launch TensorBoard

In [ ]:
import os
os.environ['TENSORBOARD_BINARY'] = '/usr/local/bin/tensorboard'

%load_ext tensorboard
# Logdir di repo (yang akan di-commit ke branch colab)
%tensorboard --logdir /content/Thesis/3DCNN/runs/v5_lowdata

## 3. Configuration

In [ ]:
import subprocess
import os
import sys
import gc
import torch
import torch.nn.functional as F

SEEDS = [42, 123, 2024, 7, 31337, 0, 1, 2, 3, 4]
VARIANTS = ["no_geom", "with_geom"]

FLAGS = {
    "no_geom":   "",
    "with_geom": "--use-geom",
}

DATA_DIR = PROJECT_ROOT / "dataset"
RUNS_DIR = PROJECT_ROOT / "runs" / "v5_lowdata"
EVAL_DIR = PROJECT_ROOT / "eval_results" / "v5_lowdata"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# v5.0.0 — Fixed budget (no early stop)
EPOCHS = 120
FINETUNE_EPOCHS = 30
TRIPLET_MARGIN = 0.3

# ============================================================
# DYNAMIC VRAM PROBE — ALWAYS target ~90% utilization
# ============================================================
# Strategy: probe forward+backward dengan BS naik bertahap.
# Stop saat actual measured VRAM ≈ 90% × total VRAM, lalu
# pakai BS terakhir yang muat.
#
# Hasil: BS optimal per-GPU per-arsitektur per-precision.
# Bekerja konsisten di T4, V100, A100 40/80, H100, dst.
# ============================================================

# --- 1. Setup constants ---
TARGET_VRAM_FRACTION = 0.75  # 75% of total VRAM (conservative: probe isolasi tidak menangkap fragmentation + real-loop overhead)
N_POINTS = 16384             # 2× default; bisa naikkan jika perlu lebih banyak compute
# SAFETY CAP untuk ball_query distance matrix: O(B * S * N) memory.
# Untuk N=16384, S=512 (SA1), matrix = B * 512 * 16384 elems.
# Di bf16: B=1024 → ~17 GB contiguous allocation — sangat berisiko OOM karena fragmentation.
# Maka untuk N_POINTS > 8192, kita hard-cap BS ≤ 192 regardless of GPU class.
MAX_BS_FOR_LARGE_N = 192 if N_POINTS > 8192 else 99999
AMP_MODE = "bf16" if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else "fp16"
PRELOAD_AUGMENT = True
NUM_WORKERS = 8

# --- 2. Detect GPU info ---
try:
    out = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=memory.total,name', '--format=csv,noheader,nounits'],
        text=True
    )
    line = out.strip().split('\n')[0]
    vram_str, gpu_name = [x.strip() for x in line.split(',')]
    VRAM_GB = int(vram_str) / 1024
    GPU_NAME = gpu_name
except Exception as e:
    print(f'GPU detection failed: {e}')
    VRAM_GB = 0
    GPU_NAME = 'Unknown'

bf16_ok = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
target_vram_gb = VRAM_GB * TARGET_VRAM_FRACTION

print(f'GPU            : {GPU_NAME} ({VRAM_GB:.1f} GB)')
print(f'BF16 support   : {bf16_ok}')
print(f'AMP mode       : {AMP_MODE}')
print(f'Target VRAM    : {target_vram_gb:.1f} GB ({TARGET_VRAM_FRACTION*100:.0f}% of {VRAM_GB:.0f} GB)')
print(f'N_POINTS       : {N_POINTS}')

# --- 3. Probe function: cari max BS untuk dapat ~90% VRAM ---
def probe_max_batch_size(n_points: int, target_gb: float, amp_dtype, use_geom: bool):
    """
    Binary search untuk BS yang pakai ~target_gb VRAM.
    Bangun model real (SiamesePalmNet), forward + backward dengan dummy data,
    ukur torch.cuda.max_memory_allocated, scale BS.
    """
    sys.path.insert(0, str(PROJECT_ROOT))
    from models.siamese import SiamesePalmNet

    device = torch.device('cuda')

    def _try_bs(bs: int) -> float:
        """Return peak memory GB or float('inf') on OOM."""
        torch.cuda.empty_cache()
        gc.collect()
        torch.cuda.reset_peak_memory_stats()
        try:
            model = SiamesePalmNet(
                geom_dim=13,
                use_geom=use_geom,
                use_aux_loss=True,
                n_subjects=10,
                siamese_mode="concat",
            ).to(device)
            model.train()
            opt = torch.optim.Adam(model.parameters(), lr=1e-3, fused=True)

            pts = torch.randn(bs, n_points, 6, device=device)
            geom = torch.randn(bs, 13, device=device)

            # Triplet mode: single-frame encode (sesuai training real)
            ctx = torch.cuda.amp.autocast(dtype=amp_dtype) if amp_dtype else torch.cuda.amp.autocast(enabled=False)
            with ctx:
                emb = model.encode(pts, geom)
                # Dummy loss (sum dari embedding)
                loss = emb.sum()
            loss.backward()
            opt.step()
            torch.cuda.synchronize()
            peak_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

            del model, opt, pts, geom, emb, loss
            torch.cuda.empty_cache()
            gc.collect()
            return peak_gb
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            gc.collect()
            return float('inf')
        except RuntimeError as e:
            if 'out of memory' in str(e).lower():
                torch.cuda.empty_cache()
                gc.collect()
                return float('inf')
            raise

    amp_dtype = torch.bfloat16 if AMP_MODE == 'bf16' else (torch.float16 if AMP_MODE == 'fp16' else None)

    # Binary search BS: mulai dari 32, double sampai OOM atau hit target, lalu refine
    print(f'\n  Probing max BS (target {target_gb:.1f} GB)...')

    bs_low, bs_high = 32, None
    last_peak = 0.0

    # Phase 1: doubling sampai OOM atau over-target
    bs = 32
    while True:
        peak = _try_bs(bs)
        if peak == float('inf'):
            print(f'    BS={bs:4d} → OOM')
            bs_high = bs
            break
        print(f'    BS={bs:4d} → {peak:.1f} GB ({100*peak/VRAM_GB:.0f}%)')
        last_peak = peak
        if peak >= target_gb * 0.95:
            # Sudah dekat target — refine
            bs_high = int(bs * (target_gb / peak) * 1.1)
            break
        bs_low = bs
        bs *= 2

    # Phase 2: binary search refine antara bs_low dan bs_high
    if bs_high is None:
        bs_high = bs_low * 2

    best_bs = bs_low
    best_peak = last_peak
    while bs_high - bs_low > 16:
        bs_mid = (bs_low + bs_high) // 2
        peak = _try_bs(bs_mid)
        if peak == float('inf') or peak > VRAM_GB * 0.95:
            print(f'    BS={bs_mid:4d} → OOM/over-budget')
            bs_high = bs_mid
        else:
            print(f'    BS={bs_mid:4d} → {peak:.1f} GB ({100*peak/VRAM_GB:.0f}%)')
            if peak <= target_gb:
                best_bs = bs_mid
                best_peak = peak
                bs_low = bs_mid
            else:
                bs_high = bs_mid

    # Conservative: kurangi sedikit untuk safety margin
    final_bs = max(32, int(best_bs * 0.90))
    # Hard cap untuk N besar agar ball_query distance matrix tidak meledak
    if N_POINTS > 8192:
        final_bs = min(final_bs, MAX_BS_FOR_LARGE_N)
        print(f'  [N-cap] BS dikunci ≤ {MAX_BS_FOR_LARGE_N} karena N_POINTS={N_POINTS} > 8192')
    print(f'  → Chosen BS = {final_bs} (peak ~{best_peak:.1f} GB, {100*best_peak/VRAM_GB:.0f}% VRAM)')
    return final_bs, best_peak


# --- 4. Probe untuk both variants (no_geom & with_geom) ---
print('\n=== PROBE no_geom ===')
BS_no_geom, peak_no_geom = probe_max_batch_size(N_POINTS, target_vram_gb, amp_dtype=None if AMP_MODE == 'none' else (torch.bfloat16 if AMP_MODE == 'bf16' else torch.float16), use_geom=False)

print('\n=== PROBE with_geom ===')
BS_with_geom, peak_with_geom = probe_max_batch_size(N_POINTS, target_vram_gb, amp_dtype=None if AMP_MODE == 'none' else (torch.bfloat16 if AMP_MODE == 'bf16' else torch.float16), use_geom=True)

# --- 5. Compute REPEAT dari BS (target dataset_len ≥ 2× BS untuk 2 steps/epoch) ---
def compute_repeat(bs: int, train_frames: int = 80, min_steps: int = 4) -> int:
    """REPEAT supaya dataset_len = train_frames * REPEAT >= bs * min_steps."""
    # Clamp bs untuk perhitungan repeat agar tidak absurd (e.g. bs capped tapi repeat tetap tinggi)
    bs_eff = min(bs, 512)
    return max(4, -(-bs_eff * min_steps // train_frames))  # ceiling division

REPEAT_no_geom = compute_repeat(BS_no_geom)
REPEAT_with_geom = compute_repeat(BS_with_geom)

# Per-variant config dict
VARIANT_CONFIG = {
    "no_geom":   {"bs": BS_no_geom,   "repeat": REPEAT_no_geom,   "peak_gb": peak_no_geom},
    "with_geom": {"bs": BS_with_geom, "repeat": REPEAT_with_geom, "peak_gb": peak_with_geom},
}

# RAM estimates
def ram_mb(bs, repeat):
    return (80 * repeat * N_POINTS * 6 * 4) / (1024 ** 2)

print('\n' + '=' * 60)
print('AUTO-TUNE RESULT (target 90% VRAM)')
print('=' * 60)
print(f'  N_POINTS        : {N_POINTS}')
print(f'  AMP_MODE        : {AMP_MODE}')
print(f'  PRELOAD_AUGMENT : {PRELOAD_AUGMENT}')
print(f'  NUM_WORKERS     : {NUM_WORKERS}')
print()
for v in VARIANTS:
    cfg = VARIANT_CONFIG[v]
    ds_len = 80 * cfg['repeat']
    steps = max(1, ds_len // cfg['bs'])
    print(f'  {v:12s}: BS={cfg["bs"]:4d}  REPEAT={cfg["repeat"]:2d}  '
          f'dataset_len={ds_len:4d}  steps/ep={steps}  '
          f'VRAM={cfg["peak_gb"]:.1f}GB ({100*cfg["peak_gb"]/VRAM_GB:.0f}%)  '
          f'RAM={ram_mb(cfg["bs"], cfg["repeat"]):.0f}MB')
print()
print(f'RUNS_DIR        : {RUNS_DIR}')
print(f'EVAL_DIR        : {EVAL_DIR}')
print(f'SEEDS           : {SEEDS}')
print(f'VARIANTS        : {VARIANTS}')

# Cleanup probe state
torch.cuda.empty_cache()
gc.collect()

## 4. Cleanup (Optional)

Set `RUN_CLEANUP = True` untuk hapus runs/eval lama sebelum mulai.

In [ ]:
import shutil

RUN_CLEANUP = False

if not RUN_CLEANUP:
    print("Skipped cleanup (RUN_CLEANUP = False)")
else:
    for d in [RUNS_DIR, EVAL_DIR]:
        if d.exists():
            shutil.rmtree(d)
            print(f"Removed: {d}")
        d.mkdir(parents=True, exist_ok=True)
    import os
    for root, dirs, files in os.walk(PROJECT_ROOT):
        for d in dirs:
            if d == '__pycache__':
                shutil.rmtree(os.path.join(root, d), ignore_errors=True)
    print("Cleanup selesai. Restart runtime sebelum lanjut.")

## 5. Gate 0 — Sanity Baseline (Geom-only LogReg)

In [ ]:
!python utils/sanity_geom_only_cv.py --dataset_root dataset

## 6. Audit Temporal Gap (Dokumentasi Limitation)

In [ ]:
import datetime
TS = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
AUDIT_DIR = PROJECT_ROOT / "result_docs" / f"{TS}_v5_lowdata"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

!python utils/audit_temporal_gap.py \
    --dataset_root dataset \
    --output {AUDIT_DIR}/temporal_gap_audit.json

print(f'\nAudit disimpan di: {AUDIT_DIR}/temporal_gap_audit.json')

## 7. Helper Functions

In [ ]:
def run_training(variant: str, seed: int) -> bool:
    out_dir = RUNS_DIR / variant / f"seed_{seed}"
    ckpt = out_dir / "best.pth"
    if ckpt.exists():
        print(f"  SKIP training {variant} seed={seed} (already done)")
        return True

    flag = FLAGS[variant]
    cfg = VARIANT_CONFIG[variant]
    bs = cfg["bs"]
    repeat = cfg["repeat"]

    amp_flag = f"--amp {AMP_MODE}" if AMP_MODE != "none" else ""
    preload_flag = "--preload-augment" if PRELOAD_AUGMENT else ""

    cmd = (
        f"python {PROJECT_ROOT / 'train.py'} "
        f"--data_dir {DATA_DIR} "
        f"--output_dir {out_dir} "
        f"--seed {seed} "
        f"--fixed_split "
        f"--frames-per-session 1 "
        f"--loss triplet --triplet-margin {TRIPLET_MARGIN} "
        f"--val-metric pair_eer --no-early-stop "
        f"--use-aux-loss "
        f"--epochs {EPOCHS} --finetune_epochs {FINETUNE_EPOCHS} "
        f"--batch_size {bs} --n_points {N_POINTS} "
        f"--num_workers {NUM_WORKERS} "
        f"--repeat {repeat} "
        f"{amp_flag} {preload_flag} "
        f"--siamese-mode concat "
        f"{flag}"
    )
    print(f"\n{'='*60}")
    print(f"START: {variant} | seed {seed} | BS={bs} N={N_POINTS} repeat={repeat} (target ~{cfg['peak_gb']:.0f}GB)")
    print(f"{'='*60}")
    try:
        mem = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=memory.used,memory.total,utilization.gpu',
             '--format=csv,noheader,nounits'],
            text=True
        ).strip().split(', ')
        print(f"  [GPU pre]  VRAM {mem[0]}/{mem[1]} MB, util {mem[2]}%")
    except Exception:
        pass
    !{cmd}
    try:
        mem = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=memory.used,memory.total,utilization.gpu',
             '--format=csv,noheader,nounits'],
            text=True
        ).strip().split(', ')
        print(f"  [GPU post] VRAM {mem[0]}/{mem[1]} MB, util {mem[2]}%")
    except Exception:
        pass
    return ckpt.exists()


def run_eval(variant: str, seed: int, split: str) -> bool:
    out_dir = RUNS_DIR / variant / f"seed_{seed}"
    ckpt = out_dir / "best.pth"
    eval_out = EVAL_DIR / variant / f"seed_{seed}" / split
    result_file = eval_out / "results.json"

    if not ckpt.exists():
        print(f"  SKIP eval {variant} seed={seed} split={split} (checkpoint missing)")
        return False
    if result_file.exists():
        print(f"  SKIP eval {variant} seed={seed} split={split} (already done)")
        return True

    eval_out.mkdir(parents=True, exist_ok=True)
    cmd = (
        f"python {PROJECT_ROOT / 'evaluate.py'} "
        f"--data_dir {DATA_DIR} "
        f"--frames-per-session 1 "
        f"--eval-split {split} "
        f"--checkpoints \"{variant}={ckpt}\" "
        f"--output_dir {eval_out} "
        f"--save_scores"
    )
    print(f"  EVAL: {variant} | seed {seed} | {split}")
    !{cmd}
    return result_file.exists()

print("Helpers ready (per-variant BS dari probe dinamis)")

## Runtime Shutdown Guard

Proteksi compute: **atexit** auto-shutdown jika crash + explicit shutdown di cell terakhir.

In [ ]:
import atexit

_shutdown_triggered = False

def shutdown_colab(silent=False):
    """Shutdown Colab runtime untuk hemat compute."""
    global _shutdown_triggered
    if _shutdown_triggered:
        return
    _shutdown_triggered = True
    if not silent:
        print('\n' + '=' * 60)
        print('\U0001f50c Shutting down Colab runtime...')
        print('=' * 60)
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        if not silent:
            print(f'  Shutdown error: {e}')
            print('  Manual: Runtime > Disconnect and delete runtime')

# Register atexit — auto-fire jika kernel crash / unexpected exit
atexit.register(shutdown_colab, silent=True)
print('\u26a1 Runtime shutdown guard terdaftar (atexit)')
print('   \u2192 Auto-shutdown jika crash')
print('   \u2192 Explicit shutdown di cell terakhir setelah selesai')

## Git Save Helper

Fungsi untuk commit + push hasil ke branch `colab`.

In [ ]:
import subprocess, os

def git_save(message: str, push: bool = False):
    """Commit semua perubahan dan opsional push ke GitHub."""
    repo = str(REPO_DIR)
    os.chdir(repo)

    # Cari file > 95MB (GitHub limit 100MB) — skip dir yang belum ada
    find_dirs = [d for d in ['3DCNN/runs', '3DCNN/eval_results', '3DCNN/analysis']
                 if os.path.isdir(os.path.join(repo, d))]
    large = ''
    if find_dirs:
        try:
            large = subprocess.check_output(
                ['find'] + find_dirs + ['-size', '+95M', '-type', 'f'],
                text=True, stderr=subprocess.DEVNULL, cwd=repo
            ).strip()
        except subprocess.CalledProcessError:
            large = ''
    if large:
        print('\u26a0\ufe0f File > 95MB (skip dari git — file besar tidak di-track):')
        for f in large.split('\n'):
            print(f'  {f}')
            subprocess.run(['git', 'update-index', '--skip-worktree', f],
                           capture_output=True)

    subprocess.run(['git', 'add', '-A'], cwd=repo)
    result = subprocess.run(['git', 'diff', '--cached', '--quiet'], cwd=repo)
    if result.returncode == 0:
        print(f'  (nothing to commit)')
        return

    subprocess.run(['git', 'commit', '-m', message], cwd=repo)
    print(f'\u2705 Committed: {message}')

    if push:
        ret = subprocess.run(
            ['git', 'push', 'origin', COLAB_BRANCH],
            cwd=repo, capture_output=True, text=True
        )
        if ret.returncode == 0:
            print(f'\u2705 Pushed ke origin/{COLAB_BRANCH}')
        else:
            print(f'\u274c Push error: {ret.stderr}')
            print('Coba push manual dari cell terakhir')

    os.chdir(str(PROJECT_ROOT))

print('git_save() ready')


## 8. no_geom — Training (10 seeds × PointNet++ murni)

In [ ]:
variant = "no_geom"
for seed in SEEDS:
    run_training(variant, seed)

## 9. no_geom — Test + Holdout Evaluation

In [ ]:
variant = "no_geom"
for seed in SEEDS:
    run_eval(variant, seed, "test")
    run_eval(variant, seed, "holdout")

In [ ]:
# Simpan hasil no_geom ke git (incremental)
git_save('v5.0.0: no_geom training + eval complete', push=True)


## 10. with_geom — Training (10 seeds × GeoAtt full)

In [ ]:
variant = "with_geom"
for seed in SEEDS:
    run_training(variant, seed)

## 11. with_geom — Test + Holdout Evaluation

In [ ]:
variant = "with_geom"
for seed in SEEDS:
    run_eval(variant, seed, "test")
    run_eval(variant, seed, "holdout")

In [ ]:
# Simpan hasil with_geom ke git (incremental)
git_save('v5.0.0: with_geom training + eval complete', push=True)


## 12. Quick Summary (Smoke check sebelum compare notebook)

In [ ]:
import json
import numpy as np

print("="*60)
print(f"{'Variant':<12} {'Seed':<6} {'Test EER':<12} {'Holdout EER':<12}")
print("="*60)
summary = {v: {'test': [], 'holdout': []} for v in VARIANTS}

for variant in VARIANTS:
    for seed in SEEDS:
        row = [f"{variant:<12}", f"{seed:<6}"]
        for split in ['test', 'holdout']:
            f = EVAL_DIR / variant / f'seed_{seed}' / split / 'results.json'
            if f.exists():
                with open(f) as fp:
                    r = json.load(fp)
                eer = r[0]['eer']
                summary[variant][split].append(eer)
                row.append(f"{eer:.4f}     ")
            else:
                row.append("missing     ")
        print(" ".join(row))

print("\n" + "="*60)
print(f"{'Variant':<12} {'Test EER (mean±std)':<25} {'Holdout EER (mean±std)':<25}")
print("="*60)
for v in VARIANTS:
    t = summary[v]['test']
    h = summary[v]['holdout']
    if t and h:
        print(f"{v:<12} {np.mean(t):.4f}±{np.std(t):.4f}              {np.mean(h):.4f}±{np.std(h):.4f}")
print("="*60)
print("\nTraining + eval selesai. Lanjut ke notebook v5_lowdata_compare.ipynb")

## Git Push Results

Commit + push semua hasil ke branch `colab`.  
Di lokal: `git fetch origin && git checkout colab` untuk lihat hasil.

In [ ]:
# Final push — semua results ke GitHub branch 'colab'
git_save('v5.0.0 low-data: all results (training + eval + analysis)', push=True)

print('\n' + '=' * 60)
print('Untuk akses di lokal:')
print('  git fetch origin')
print('  git checkout colab')
print('  # atau: git pull origin colab')
print('=' * 60)


## Auto-Shutdown Compute

Otomatis shutdown runtime setelah semua proses selesai.  
Delay 60 detik untuk review output. Cancel: **Runtime > Interrupt execution**.

In [ ]:
import time

SHUTDOWN_DELAY = 60  # detik, beri waktu review hasil

print('=' * 60)
print('\u2705 ALL DONE — v5.0.0 Low-Data Training + Evaluation selesai!')
print('=' * 60)
print(f'\nRuntime akan di-shutdown dalam {SHUTDOWN_DELAY} detik.')
print('Untuk membatalkan: Runtime > Interrupt execution\n')

try:
    for remaining in range(SHUTDOWN_DELAY, 0, -10):
        print(f'  Shutdown dalam {remaining} detik...')
        time.sleep(min(10, remaining))
    shutdown_colab()
except KeyboardInterrupt:
    print('\n\u26a0\ufe0f Shutdown dibatalkan. Runtime tetap aktif.')
    _shutdown_triggered = False  # reset supaya bisa dipanggil lagi